# Data Cleaning
**Topic:** Scientific Python

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


---
## What you'll explore

By the end of this demo you will be able to:

- **Identify** missing values, duplicate rows, and type mismatches in a realistic messy dataset
- **Apply** imputation, deduplication, and type conversion strategies using pandas
- **Interpret** the trade-offs between dropping missing values and imputing them

---
## How we got here

In *14: Pandas* we loaded and explored clean data. Real data is never clean. Missing values, duplicate rows, inconsistent formats, and out-of-range values are the norm, not the exception. Data scientists typically spend 60-80% of project time on cleaning, and this notebook introduces the systematic process for doing it repeatably and auditably.

---
## Why this matters for data science

Scikit-learn's estimators raise a `ValueError` if they receive `NaN` values. Most ML algorithms are also sensitive to outliers and inconsistent encodings. A model trained on dirty data learns the dirt as signal; a model trained on clean data learns the actual patterns. Getting cleaning right, and documenting every decision in code rather than by hand in Excel, is one of the most important practices that separates reproducible data science from ad hoc analysis.

---
## Try it yourself

In [ ]:
# ▶ Run this cell and observe the output.
# Then try changing the values and running again.

import pandas as pd

df = pd.DataFrame({
    'Name ':  ['Alice', 'Bob', 'Alice', None],
    ' Age':   [25, None, 25, 30],
    'Salary$': ['$72,000', '$85,000', '$72,000', 'n/a'],
})

print('Before cleaning:')
print(df)
print('\nDtypes:', df.dtypes.to_dict())
print('Missing values:\n', df.isna().sum())

In [ ]:
# ✏️ Your turn — modify this code:
# 1. Change fillna(method) to use the median instead of mean for numeric columns
# 2. Add .drop_duplicates() before printing the result
# 3. What does df.info() show that df.describe() doesn't?

import pandas as pd
import numpy as np

df = pd.DataFrame({
    'score':  [88.0, None, 72.0, 91.0, None],
    'grade':  ['B', 'C', None, 'A', 'B'],
})

df['score'] = df['score'].fillna(df['score'].mean())
df['grade'] = df['grade'].fillna('Unknown')

print(df)

In [ ]:
# 🎯 Challenge:
# Given this messy DataFrame, apply a full cleaning pipeline:
#   1. Standardize column names (lowercase, replace spaces with underscores)
#   2. Handle missing values appropriately (numeric → median, categorical → 'Unknown')
#   3. Remove duplicate rows
#   4. Fix the 'price' column so it is a float (remove '$' and ',')
# Hint: use .str.strip().str.lower().str.replace() for column name cleanup

import pandas as pd

messy = pd.DataFrame({
    ' Product Name ': ['Widget', 'Gadget', 'Widget', None, 'Doohickey'],
    'Unit Price':     ['$12,50', '$8,99', '$12,50', '$5,00', None],
    'Category ':      ['Tools', None, 'Tools', 'Electronics', 'Electronics'],
    'In Stock':       [True, True, True, False, None],
})

# Your code here:

---
## What's happening?

Data cleaning is the process of detecting and correcting problems in raw data so that downstream analysis and modeling produce valid results. The most common problems form a short checklist.

| Problem | Detection | Fix |
|---------|-----------|-----|
| Missing values | `df.isna().sum()` | Drop, impute with mean/median/mode, or forward-fill |
| Duplicate rows | `df.duplicated().sum()` | `df.drop_duplicates()` |
| Wrong dtypes | `df.dtypes` | `pd.to_numeric()`, `pd.to_datetime()`, `.astype()` |
| Outliers | `.describe()`, box plots | Clip, log-transform, or investigate individually |
| Inconsistent strings | `.value_counts()` | `.str.strip().str.lower()`, `.replace()` |
| Impossible values | Domain knowledge checks | Set to `NaN`, then impute |

```python
def clean_dataframe(df):
    """Reproducible cleaning pipeline for the housing dataset."""
    df = df.drop_duplicates()
    df["price"]    = pd.to_numeric(df["price"], errors="coerce")
    df["zip_code"] = df["zip_code"].astype(str).str.zfill(5)
    numeric_cols   = df.select_dtypes("number").columns
    df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())
    df["neighborhood"] = df["neighborhood"].str.strip().str.title()
    df["price"]    = df["price"].clip(upper=df["price"].quantile(0.99))
    return df
```

### Imputing vs dropping: the core trade-off

Dropping rows with missing values is safe when missingness is rare (< 5%) and random. Imputing with the mean or median preserves all rows but shrinks variance and can bias model coefficients. More sophisticated strategies (k-NN imputation, model-based imputation) are available in `sklearn.impute`.

Return to the widget and compare the before/after row counts and distributions for the mean imputation vs. drop strategy.

---
## A direct example: filling missing values with the median

Six student scores, two of them missing. `fillna()` replaces both `NaN` values with the column median — the same strategy scikit-learn's `SimpleImputer` uses by default.

- **Notice:** The median is computed from the four non-null values only — missing values do not contribute to the calculation
- **Notice:** Red bars show the imputed rows; blue bars are unchanged originals — the chart makes it visible exactly which rows were affected
- **Notice:** Median imputation is preferred over mean when outliers are present, because outliers skew the mean but not the median

In [ ]:
df = pd.DataFrame({
    "name":  ["Alice", "Bob", "Carol", "David", "Eve", "Frank"],
    "score": [88.0, None, 72.0, None, 91.0, 65.0],
})

was_missing = df["score"].isna().tolist()
fill_value  = df["score"].median()
df["score"] = df["score"].fillna(fill_value)

print(df)

colors = ["#E45756" if m else "#4C72B0" for m in was_missing]

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(df["name"], df["score"], color=colors)
ax.axhline(fill_value, linestyle="--", color="black", linewidth=1,
           label=f"fill value (median) = {fill_value:.0f}")
ax.set_title("Missing scores (red) filled with the column median", fontsize=13)
ax.set_xlabel("Student")
ax.set_ylabel("Score")
ax.legend()
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

---
## Real-world example: Cleaning a messy housing price dataset

The chart shows the price column distribution from a simulated housing dataset before and after a five-step cleaning pipeline. This mirrors the kind of messy data that appears on Kaggle's Ames Housing dataset and in real real-estate analytics.

Notice:

- **Notice:** The raw distribution has a long right tail and several implausible values above $2M that are almost certainly data-entry errors; clipping at the 99th percentile removes them without discarding entire rows
- **Notice:** Missing values in the price column (shown by the gap in the raw histogram around $0) are imputed with the median, which avoids pulling the mean toward the high outliers
- **Notice:** The cleaned distribution is still right-skewed, that is a real property of housing prices, not an artifact, so a log transform might still be appropriate before modeling

> **Discussion question:** After clipping outliers at the 99th percentile, a colleague says the model "can never predict a house worth more than the 99th percentile." Is that true? What actually happens when the model receives a feature value that was clipped in training?

In [ ]:
np.random.seed(88)

# ── Messy housing price dataset: before and after cleaning ────────────────────
n            = 400
prices_raw   = np.random.lognormal(mean=12.5, sigma=0.45, size=n)
prices_raw[np.random.choice(n, 10, replace=False)] *= np.random.uniform(5, 15, 10)
prices_series = pd.Series(prices_raw)
prices_series.iloc[np.random.choice(n, 20, replace=False)] = np.nan

median_price   = prices_series.median()
prices_imputed = prices_series.fillna(median_price)
cap_99         = prices_imputed.quantile(0.99)
prices_clean   = prices_imputed.clip(upper=cap_99)

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(prices_series.dropna() / 1000, bins=45, density=True, range=(0, 900),
        color="#E45756", alpha=0.6, label="Raw (NaNs dropped for display)")
ax.hist(prices_clean / 1000, bins=45, density=True, range=(0, 900),
        color="#4C72B0", alpha=0.6, label="After cleaning pipeline")
ax.set_title("Housing Prices: Before vs After Cleaning Pipeline", fontsize=13)
ax.set_xlabel("Price ($000s)")
ax.set_ylabel("Probability Density")
ax.legend()
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()


### The cleaning decision guide

| Missing value % | Imputation strategy | Why |
|----------------|--------------------|----|
| < 5%, random | Drop rows | Minimal data loss, no bias introduced |
| 5-30%, random | Impute with median (numeric) or mode (categorical) | Preserves rows; median is outlier-resistant |
| 5-30%, not random | Investigate cause; consider model-based imputation | Non-random missingness can bias simple imputation |
| > 30% | Evaluate dropping the column entirely | Imputing half a column may introduce more noise than signal |
| Systematic (e.g., "No answer") | Create a binary `is_missing` indicator feature | Missingness itself may be predictive |

---
## Key takeaway

> **Data cleaning is not cosmetic; every missing value left unaddressed, every outlier left in, and every type mismatch left unfixed will show up as wrong predictions or crashing code downstream in the pipeline.**

---
*Next up: Visualization — choosing the right chart to reveal the patterns that cleaning has uncovered, and building the plots that communicate your findings to others*